In [1]:
!git clone https://github.com/kalyaannnn/agentRL.git
%cd /content/agentRL
!pip install -e ".[benchmark]"
!pip install datasets


Cloning into 'agentRL'...
remote: Enumerating objects: 333, done.
remote: Counting objects: 100% (333/333), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 333 (delta 200), reused 252 (delta 123), pack-reused 0 (from 0)
Receiving objects: 100% (333/333), 185.90 KiB | 16.90 MiB/s, done.
Resolving deltas: 100% (200/200), done.
/content/agentRL
Obtaining file:///content/agentRL
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for agentrl (pyproject.toml) ... done
  Created wheel for agentrl: filename=agentrl-0.1.0-py3-none-any.whl size=7919 sha256=cb04aa1f42cbd59661196ee3fb8b9ecf903020bffd24165e4b2c7ca3ca08a1be
  Stored in directory: /tmp/pip-ephem-wheel-cache-cfxkctib/wheels/b1/19/46/57c2b6a75da3ae2ee8cd8162ce3c1f8e208a5ef11c2ac3f011
Successfully 

In [2]:
from __future__ import annotations

import ast
import json
import subprocess
import sys
import tempfile
import textwrap
from dataclasses import dataclass
from pathlib import Path

from datasets import load_dataset

from agentrl import BYODRecord, GRPOConfig, GRPOTrainer, SFTBootstrapTrainer, make_single_turn_task
from agentrl.memory.layout import SharedWeightLayout


In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
SUBSET_SIZE = 32
BOOTSTRAP_EPOCHS = 2
RL_STEPS = 5
BATCH_SIZE = 1
GROUP_SIZE = 2
MAX_NEW_TOKENS = 128

BOOTSTRAP_DIR = "./mbpp_bootstrap_adapter"
RL_DIR = "./mbpp_rl_run"


In [4]:
raw_dataset = load_dataset("mbpp", "sanitized", split=f"test[:{SUBSET_SIZE}]")

@dataclass(frozen=True)
class MBPPExample:
    task_id: int
    prompt: str
    reference_code: str
    test_setup_code: str
    test_list: list[str]

examples = [
    MBPPExample(
        task_id=int(row["task_id"]),
        prompt=str(row["prompt"]),
        reference_code=str(row["code"]),
        test_setup_code="\n".join(row.get("test_imports", [])),
        test_list=list(row["test_list"]),
    )
    for row in raw_dataset
]

example_by_task_id = {example.task_id: example for example in examples}
len(examples), examples[0]


README.md: 0.00B [00:00, ?B/s]

sanitized/train-00000-of-00001.parquet:   0%|          | 0.00/33.9k [00:00<?, ?B/s]

sanitized/test-00000-of-00001.parquet:   0%|          | 0.00/60.9k [00:00<?, ?B/s]

sanitized/validation-00000-of-00001.parq(…):   0%|          | 0.00/14.0k [00:00<?, ?B/s]

sanitized/prompt-00000-of-00001.parquet:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/257 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/43 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/7 [00:00<?, ? examples/s]

(32,
 MBPPExample(task_id=11, prompt='Write a python function to remove first and last occurrence of a given character from the string.', reference_code='def remove_Occ(s,ch): \n    for i in range(len(s)): \n        if (s[i] == ch): \n            s = s[0 : i] + s[i + 1:] \n            break\n    for i in range(len(s) - 1,-1,-1):  \n        if (s[i] == ch): \n            s = s[0 : i] + s[i + 1:] \n            break\n    return s ', test_setup_code='', test_list=['assert remove_Occ("hello","l") == "heo"', 'assert remove_Occ("abcda","a") == "bcd"', 'assert remove_Occ("PHP","P") == "H"']))

In [5]:
def make_prompt(example: MBPPExample) -> str:
    tests_preview = "\n".join(example.test_list[:3])
    return textwrap.dedent(f"""
    Write Python code that solves the problem below.

    Problem:
    {example.prompt}

    Requirements:
    - Return only Python code.
    - Do not include markdown fences.
    - Do not include print statements.
    - Do not include example usage.
    - Define exactly the function needed by the tests.

    Tests your code must satisfy:
    {tests_preview}
    """).strip()


def strip_code_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    return text


def truncate_generated_code(code: str) -> str:
    lines = strip_code_fences(code).splitlines()
    kept = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith("print("):
            break
        if stripped.startswith("# Example"):
            break
        if stripped.startswith("# Test"):
            break
        if stripped.startswith("if __name__ == "):
            break
        kept.append(line)
    return "\n".join(kept).strip()


In [6]:
def run_python_with_test_stats(code: str, setup_code: str, tests: list[str], timeout_seconds: int = 3):
    code = truncate_generated_code(code)

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        candidate_path = tmp / "candidate.py"

        script_lines = [
            "import math",
            "import itertools",
            "import functools",
            "import collections",
            "import heapq",
            "import bisect",
            "import re",
            "import string",
            "import typing",
            "",
        ]

        if setup_code.strip():
            script_lines.append(setup_code)
            script_lines.append("")

        script_lines.append(code)
        script_lines.append("")
        script_lines.append("passed_count = 0")
        script_lines.append(f"total_count = {len(tests)}")
        script_lines.append("status = 'ok'")
        script_lines.append("")

        for test in tests:
            indented = "\n".join("    " + line for line in test.splitlines())
            script_lines.append("try:")
            script_lines.append(indented)
            script_lines.append("    passed_count += 1")
            script_lines.append("except AssertionError:")
            script_lines.append("    pass")
            script_lines.append("except Exception:")
            script_lines.append("    pass")
            script_lines.append("")

        script_lines.append("print({'passed_count': passed_count, 'total_count': total_count, 'status': status})")
        candidate_path.write_text("\n".join(script_lines), encoding="utf-8")

        try:
            result = subprocess.run(
                [sys.executable, str(candidate_path)],
                cwd=tmp,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=timeout_seconds,
            )
        except subprocess.TimeoutExpired:
            return {
                "passed_count": 0,
                "total_count": len(tests),
                "status": "timeout",
                "stdout": "",
                "stderr": "Timed out",
            }

        stdout = (result.stdout or "").strip()
        stderr = (result.stderr or "").strip()

        if result.returncode == 0 and stdout:
            try:
                stats = ast.literal_eval(stdout)
                return {
                    "passed_count": int(stats["passed_count"]),
                    "total_count": int(stats["total_count"]),
                    "status": str(stats["status"]),
                    "stdout": stdout,
                    "stderr": stderr,
                }
            except Exception:
                pass

        lowered = stderr.lower()
        status = "runtime_error"
        if "syntaxerror" in lowered:
            status = "syntax_error"

        return {
            "passed_count": 0,
            "total_count": len(tests),
            "status": status,
            "stdout": stdout,
            "stderr": stderr,
        }


def strict_pass_fail(example: MBPPExample, generated_code: str) -> bool:
    stats = run_python_with_test_stats(
        code=generated_code,
        setup_code=example.test_setup_code,
        tests=example.test_list,
        timeout_seconds=3,
    )
    return stats["passed_count"] == stats["total_count"]


In [7]:
records = [
    BYODRecord(
        input=make_prompt(example),
        reference_answer=f"task::{example.task_id}",
        supervised_target=example.reference_code,
        metadata={"task_id": example.task_id},
    )
    for example in examples
]

records[0]


BYODRecord(input='Write Python code that solves the problem below.\n\n    Problem:\n    Write a python function to remove first and last occurrence of a given character from the string.\n\n    Requirements:\n    - Return only Python code.\n    - Do not include markdown fences.\n    - Do not include print statements.\n    - Do not include example usage.\n    - Define exactly the function needed by the tests.\n\n    Tests your code must satisfy:\n    assert remove_Occ("hello","l") == "heo"\nassert remove_Occ("abcda","a") == "bcd"\nassert remove_Occ("PHP","P") == "H"', reference_answer='task::11', supervised_target='def remove_Occ(s,ch): \n    for i in range(len(s)): \n        if (s[i] == ch): \n            s = s[0 : i] + s[i + 1:] \n            break\n    for i in range(len(s) - 1,-1,-1):  \n        if (s[i] == ch): \n            s = s[0 : i] + s[i + 1:] \n            break\n    return s ', metadata={'task_id': 11})

In [8]:
def prompt_formatter(record: BYODRecord, tokenizer) -> str:
    if tokenizer is not None and hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": "You write correct Python code that passes tests."},
            {"role": "user", "content": record.input},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return (
        "System:\nYou write correct Python code that passes tests.\n\n"
        f"User:\n{record.input}\n\nAssistant:\n"
    )


def supervised_target_fn(record: BYODRecord) -> str | None:
    return record.supervised_target


def reward_fn(response: str, state: dict[str, object]) -> float:
    task_id = int(state["metadata"]["task_id"])
    example = example_by_task_id[task_id]

    stats = run_python_with_test_stats(
        code=response,
        setup_code=example.test_setup_code,
        tests=example.test_list,
        timeout_seconds=3,
    )

    passed = stats["passed_count"]
    total = stats["total_count"]
    status = stats["status"]

    if total <= 0:
        return 0.0

    base = passed / total

    if passed == total:
        return 1.0
    if status == "syntax_error":
        return min(1.0, 0.05 + 0.25 * base)
    if status == "timeout":
        return min(1.0, 0.05 + 0.20 * base)
    if status == "runtime_error":
        return min(1.0, 0.05 + 0.30 * base)
    return min(1.0, 0.10 + 0.80 * base)


In [9]:
task = make_single_turn_task(
    records=records,
    prompt_formatter=prompt_formatter,
    reward_fn=reward_fn,
    supervised_target_fn=supervised_target_fn,
    seed=0,
)

prompt = task.environment.reset()
state = task.environment.state()
sample_prompt, sample_target = task.supervised_samples(tokenizer=None)[0]

print(prompt[:700])
print(state)
print(sample_target[:400])


System:
You write correct Python code that passes tests.

User:
Write Python code that solves the problem below.

    Problem:
    Write a function to check whether it follows the sequence given in the patterns array.

    Requirements:
    - Return only Python code.
    - Do not include markdown fences.
    - Do not include print statements.
    - Do not include example usage.
    - Define exactly the function needed by the tests.

    Tests your code must satisfy:
    assert is_samepatterns(["red","green","green"], ["a", "b", "b"])==True
assert is_samepatterns(["red","green","greenn"], ["a","b","b"])==False
assert is_samepatterns(["red","green","greenn"], ["a","b"])==False

Assistant:

{'input': 'Write Python code that solves the problem below.\n\n    Problem:\n    Write a function to check whether it follows the sequence given in the patterns array.\n\n    Requirements:\n    - Return only Python code.\n    - Do not include markdown fences.\n    - Do not include print statements.\n  

In [10]:
example = examples[0]
stats = run_python_with_test_stats(
    code=example.reference_code,
    setup_code=example.test_setup_code,
    tests=example.test_list,
)
print(stats)


{'passed_count': 3, 'total_count': 3, 'status': 'ok', 'stdout': "{'passed_count': 3, 'total_count': 3, 'status': 'ok'}", 'stderr': ''}


In [11]:
from peft import LoraConfig
from transformers import AutoTokenizer
import torch

bootstrap_config = GRPOConfig(
    model_name=MODEL_NAME,
    batch_size=2,
    max_prompt_tokens=768,
    lr=5e-5,
    steps=1,
    output_dir=BOOTSTRAP_DIR,
    use_continuous_batching=False,
    use_speculative_decoding=False,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if getattr(tokenizer, "pad_token_id", None) is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"

layout = SharedWeightLayout(
    model_name=MODEL_NAME,
    lora_config=LoraConfig(
        r=bootstrap_config.lora_r,
        lora_alpha=bootstrap_config.lora_alpha,
        lora_dropout=bootstrap_config.lora_dropout,
        target_modules=list(bootstrap_config.lora_target_modules),
        bias="none",
        task_type="CAUSAL_LM",
    ),
    dtype=bootstrap_config.dtype,
    device=device,
    trust_remote_code=bootstrap_config.trust_remote_code,
    sdpa_backend=bootstrap_config.sdpa_backend,
)

bootstrap_trainer = SFTBootstrapTrainer(
    config=bootstrap_config,
    tokenizer=tokenizer,
    layout=layout,
)

bootstrap_history = bootstrap_trainer.train(
    task.supervised_samples(tokenizer=tokenizer),
    epochs=BOOTSTRAP_EPOCHS,
)
bootstrap_trainer.save_adapter(BOOTSTRAP_DIR)

bootstrap_history[-5:]


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[{'step': 27.0,
  'epoch': 1.0,
  'loss': 0.4915159046649933,
  'batch_size': 2.0,
  'steps_per_epoch': 16.0},
 {'step': 28.0,
  'epoch': 1.0,
  'loss': 0.6079869270324707,
  'batch_size': 2.0,
  'steps_per_epoch': 16.0},
 {'step': 29.0,
  'epoch': 1.0,
  'loss': 0.7794458270072937,
  'batch_size': 2.0,
  'steps_per_epoch': 16.0},
 {'step': 30.0,
  'epoch': 1.0,
  'loss': 0.5688217878341675,
  'batch_size': 2.0,
  'steps_per_epoch': 16.0},
 {'step': 31.0,
  'epoch': 1.0,
  'loss': 1.0327186584472656,
  'batch_size': 2.0,
  'steps_per_epoch': 16.0}]

In [12]:
from transformers import AutoTokenizer
import torch

eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if getattr(eval_tokenizer, "pad_token_id", None) is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

bootstrap_eval_layout = SharedWeightLayout(
    model_name=MODEL_NAME,
    lora_config=LoraConfig(
        r=bootstrap_config.lora_r,
        lora_alpha=bootstrap_config.lora_alpha,
        lora_dropout=bootstrap_config.lora_dropout,
        target_modules=list(bootstrap_config.lora_target_modules),
        bias="none",
        task_type="CAUSAL_LM",
    ),
    dtype=bootstrap_config.dtype,
    device=device,
    trust_remote_code=bootstrap_config.trust_remote_code,
    sdpa_backend=bootstrap_config.sdpa_backend,
    adapter_path=BOOTSTRAP_DIR,
)

bootstrap_model = bootstrap_eval_layout.model
bootstrap_model.eval()

bootstrap_preview = []

for example in examples[:5]:
    record = BYODRecord(
        input=make_prompt(example),
        reference_answer=f"task::{example.task_id}",
        supervised_target=example.reference_code,
        metadata={"task_id": example.task_id},
    )
    rendered_prompt = prompt_formatter(record, eval_tokenizer)

    encoded = eval_tokenizer(
        rendered_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )
    input_ids = encoded["input_ids"].to(bootstrap_eval_layout.device)
    attention_mask = encoded["attention_mask"].to(bootstrap_eval_layout.device)

    with torch.no_grad():
        generated = bootstrap_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=eval_tokenizer.pad_token_id,
            eos_token_id=getattr(eval_tokenizer, "eos_token_id", None),
        )

    response_ids = generated[:, input_ids.shape[-1]:]
    response = eval_tokenizer.decode(response_ids[0], skip_special_tokens=True)
    response = truncate_generated_code(response)

    stats = run_python_with_test_stats(
        code=response,
        setup_code=example.test_setup_code,
        tests=example.test_list,
    )

    bootstrap_preview.append(
        {
            "task_id": example.task_id,
            "passed_count": stats["passed_count"],
            "total_count": stats["total_count"],
            "preview": response[:250],
        }
    )

bootstrap_preview


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[{'task_id': 11,
  'passed_count': 1,
  'total_count': 3,
  'preview': 'def remove_Occ(s, ch):\n  l = s.find(ch)\n  r = s.rfind(ch)\n  if (l != -1) and (r != -1):\n    return s[:l] + s[r+1:]\n  else:\n    return s[:]  # If no such characters are found, return the original string. \n  pass  # Do not include this line in your sub'},
 {'task_id': 12,
  'passed_count': 0,
  'total_count': 3,
  'preview': 'def sort_matrix(matrix):\n    row_sums = sorted([sum(row) for row in matrix])\n    return [[row[i] for i in range(len(row)) if row[i] != row[i-1]] + [row[-1]] for row in zip(*matrix)]  #zip() is used to transpose the matrix and then we use list compreh'},
 {'task_id': 14,
  'passed_count': 3,
  'total_count': 3,
  'preview': 'def find_Volume(a,b,c): \n\treturn (a*b*c)/2;  # Volume of Triangular Prism = Area * Height / 2  where A is area and h is height  in units of length  1 unit = 1  so we divide by 2  and multiply by 1  which gives us our answer 1/2 * 1 * 1 * 1  or  1  fo'},
 {'task_id'

In [13]:
rl_config = GRPOConfig(
    model_name=MODEL_NAME,
    steps=RL_STEPS,
    batch_size=BATCH_SIZE,
    group_size=GROUP_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=0.8,
    top_p=0.95,
    output_dir=RL_DIR,
    init_adapter_path=BOOTSTRAP_DIR,
)

trainer = GRPOTrainer(
    config=rl_config,
    environment=task.environment,
    verifier=task.verifier,
)

rl_history = trainer.train()
rl_history[-5:]


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

step=0 | mean_reward=0.5250 | reward_std=0.4750 | policy_loss=-0.6701 | kl_loss=0.0011 | mean_token_kl=0.1087 | beta=0.0100 | total_loss=-0.6690 | learning_rate=0.0000 | generation_time_ms=11408.6384 | prefill_time_ms=153.8744 | decode_time_ms=10371.7905 | training_time_ms=286.9741 | generation_peak_vram_mb=4715.9116 | rollout_peak_vram_mb=4715.9116 | generation_runtime_headroom_mb=17847.2134 | rollout_runtime_headroom_mb=17847.2134 | peak_vram_mb=5870.4590 | tokens_per_second=21.8886 | prefill_tokens_per_second=1884.6535 | decode_tokens_per_second=24.6823 | padding_ratio=0.0002 | padding_waste_tokens=12.0000 | cache_reuse_effectiveness=0.9946 | unique_response_ratio=1.0000 | advantage_mean=8.940696716308594e-08 | advantage_std=1.0 | training_peak_vram_mb=5870.458984375 | training_runtime_headroom_mb=16692.666015625 | total_step_time_ms=11695.612460999997 | phase_a_fraction=0.9754631010597381 | effective_batch_tokens=256.0 | generation_padding_ratio=0.0 | generation_padding_waste_token

step=1 | mean_reward=0.0500 | reward_std=0.0000 | policy_loss=-0.0000 | kl_loss=0.0010 | mean_token_kl=0.1023 | beta=0.0100 | total_loss=0.0010 | learning_rate=0.0000 | generation_time_ms=10865.1788 | prefill_time_ms=150.0393 | decode_time_ms=9894.4655 | training_time_ms=282.1473 | generation_peak_vram_mb=5870.4590 | rollout_peak_vram_mb=5870.4590 | generation_runtime_headroom_mb=16692.6660 | rollout_runtime_headroom_mb=16692.6660 | peak_vram_mb=5937.3652 | tokens_per_second=22.8754 | prefill_tokens_per_second=1932.8272 | decode_tokens_per_second=25.8730 | padding_ratio=0.0003 | padding_waste_tokens=14.0000 | cache_reuse_effectiveness=0.9946 | unique_response_ratio=1.0000 | advantage_mean=0.0 | advantage_std=0.0 | training_peak_vram_mb=5937.365234375 | training_runtime_headroom_mb=16625.759765625 | total_step_time_ms=11147.32613199999 | phase_a_fraction=0.974689240033087 | effective_batch_tokens=255.0 | generation_padding_ratio=0.0 | generation_padding_waste_tokens=0.0 | sequence_paddi

step=4 | mean_reward=0.0500 | reward_std=0.0000 | policy_loss=-0.0000 | kl_loss=0.0010 | mean_token_kl=0.1026 | beta=0.0100 | total_loss=0.0010 | learning_rate=0.0000 | generation_time_ms=10758.4549 | prefill_time_ms=154.2662 | decode_time_ms=9789.1740 | training_time_ms=281.6359 | generation_peak_vram_mb=5937.3926 | rollout_peak_vram_mb=5937.3926 | generation_runtime_headroom_mb=16625.7324 | rollout_runtime_headroom_mb=16625.7324 | peak_vram_mb=5937.4062 | tokens_per_second=23.0976 | prefill_tokens_per_second=1879.8673 | decode_tokens_per_second=26.1513 | padding_ratio=0.0003 | padding_waste_tokens=14.0000 | cache_reuse_effectiveness=0.9946 | unique_response_ratio=1.0000 | advantage_mean=0.0 | advantage_std=0.0 | training_peak_vram_mb=5937.40625 | training_runtime_headroom_mb=16625.71875 | total_step_time_ms=11040.090785000075 | phase_a_fraction=0.974489715575279 | effective_batch_tokens=255.0 | generation_padding_ratio=0.0 | generation_padding_waste_tokens=0.0 | sequence_padding_rati

[{'step': 0.0,
  'mean_reward': 0.5249999761581421,
  'reward_std': 0.4749999940395355,
  'policy_loss': -0.6701149940490723,
  'kl_loss': 0.0010870335390791297,
  'mean_token_kl': 0.1087033599615097,
  'total_loss': -0.6690279841423035,
  'advantage_mean': 8.940696716308594e-08,
  'advantage_std': 1.0,
  'unique_response_ratio': 1.0,
  'learning_rate': 1e-05,
  'beta': 0.01,
  'generation_time_ms': 11408.638399999973,
  'generation_peak_vram_mb': 4715.91162109375,
  'generation_runtime_headroom_mb': 17847.21337890625,
  'training_time_ms': 286.97406100002354,
  'training_peak_vram_mb': 5870.458984375,
  'training_runtime_headroom_mb': 16692.666015625,
  'peak_vram_mb': 5870.458984375,
  'total_step_time_ms': 11695.612460999997,
  'phase_a_fraction': 0.9754631010597381,
  'effective_batch_tokens': 256.0,
  'tokens_per_second': 21.888550159613573,
  'padding_ratio': 0.00022248590922574905,
  'padding_waste_tokens': 12.0,
  'generation_padding_ratio': 0.0,
  'generation_padding_waste_tok

In [14]:
metrics_path = Path(RL_DIR) / "metrics.jsonl"
print(metrics_path.read_text(encoding="utf-8")[:3000])


{"advantage_mean": 8.940696716308594e-08, "advantage_std": 1.0, "beta": 0.01, "cache_reuse_effectiveness": 0.9945962061640518, "cache_reuse_tokens": 53376.0, "decode_time_ms": 10371.79050599991, "decode_tokens": 256.0, "decode_tokens_per_second": 24.682334246136985, "dominant_runtime_bottleneck": "decode", "effective_batch_tokens": 256.0, "generation_padding_ratio": 0.0, "generation_padding_waste_tokens": 0.0, "generation_peak_vram_mb": 4715.91162109375, "generation_runtime_headroom_mb": 17847.21337890625, "generation_time_ms": 11408.638399999973, "kl_loss": 0.0010870335390791297, "kv_pressure_streak": 0.0, "learning_rate": 1e-05, "mean_reward": 0.5249999761581421, "mean_token_kl": 0.1087033599615097, "padding_pressure_streak": 0.0, "padding_ratio": 0.00022248590922574905, "padding_waste_tokens": 12.0, "paged_kv_allocator_occupancy": 0.0, "paged_kv_allocator_pressure": 0.0, "paged_kv_block_reuse_count": 0.0, "paged_kv_block_size_tokens": 0.0, "paged_kv_free_block_count": 0.0, "paged_kv

In [15]:
rl_eval_layout = SharedWeightLayout(
    model_name=MODEL_NAME,
    lora_config=LoraConfig(
        r=rl_config.lora_r,
        lora_alpha=rl_config.lora_alpha,
        lora_dropout=rl_config.lora_dropout,
        target_modules=list(rl_config.lora_target_modules),
        bias="none",
        task_type="CAUSAL_LM",
    ),
    dtype=rl_config.dtype,
    device=device,
    trust_remote_code=rl_config.trust_remote_code,
    sdpa_backend=rl_config.sdpa_backend,
    adapter_path=str(Path(RL_DIR) / "checkpoint_final"),
)

rl_model = rl_eval_layout.model
rl_model.eval()

final_results = []

for example in examples[:5]:
    record = BYODRecord(
        input=make_prompt(example),
        reference_answer=f"task::{example.task_id}",
        supervised_target=example.reference_code,
        metadata={"task_id": example.task_id},
    )
    rendered_prompt = prompt_formatter(record, eval_tokenizer)

    encoded = eval_tokenizer(
        rendered_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )
    input_ids = encoded["input_ids"].to(rl_eval_layout.device)
    attention_mask = encoded["attention_mask"].to(rl_eval_layout.device)

    with torch.no_grad():
        generated = rl_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=eval_tokenizer.pad_token_id,
            eos_token_id=getattr(eval_tokenizer, "eos_token_id", None),
        )

    response_ids = generated[:, input_ids.shape[-1]:]
    response = eval_tokenizer.decode(response_ids[0], skip_special_tokens=True)
    response = truncate_generated_code(response)

    stats = run_python_with_test_stats(
        code=response,
        setup_code=example.test_setup_code,
        tests=example.test_list,
    )

    final_results.append(
        {
            "task_id": example.task_id,
            "passed_count": stats["passed_count"],
            "total_count": stats["total_count"],
            "strict_pass": stats["passed_count"] == stats["total_count"],
            "response_preview": response[:300],
        }
    )

final_results


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[{'task_id': 11,
  'passed_count': 1,
  'total_count': 3,
  'strict_pass': False,
  'response_preview': 'def remove_Occ(s, ch):\n  l = s.find(ch)\n  r = s.rfind(ch)\n  if (l != -1) and (r != -1):\n    return s[:l] + s[r+1:]\n  else:\n    return s[:]  # If no such characters are found, return the original string. \n  pass  # Do not include this line in your submission. It is here for you to test your solution.'},
 {'task_id': 12,
  'passed_count': 0,
  'total_count': 3,
  'strict_pass': False,
  'response_preview': 'def sort_matrix(matrix):\n    row_sums = sorted([sum(row) for row in matrix])\n    return [[row[i] for i in range(len(row)) if row[i] != row[i-1]] + [row[-1]] for row in zip(*matrix)]  #zip() is used to transpose the matrix and then we use list comprehension to add the last element at the end of each '},
 {'task_id': 14,
  'passed_count': 3,
  'total_count': 3,
  'strict_pass': True,
  'response_preview': 'def find_Volume(a,b,c): \n\treturn (a*b*c)/2;  # Volume of Triangular

In [16]:
aggregate_results = []

for example in examples:
    record = BYODRecord(
        input=make_prompt(example),
        reference_answer=f"task::{example.task_id}",
        supervised_target=example.reference_code,
        metadata={"task_id": example.task_id},
    )
    rendered_prompt = prompt_formatter(record, eval_tokenizer)

    encoded = eval_tokenizer(
        rendered_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )
    input_ids = encoded["input_ids"].to(rl_eval_layout.device)
    attention_mask = encoded["attention_mask"].to(rl_eval_layout.device)

    with torch.no_grad():
        generated = rl_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=eval_tokenizer.pad_token_id,
            eos_token_id=getattr(eval_tokenizer, "eos_token_id", None),
        )

    response_ids = generated[:, input_ids.shape[-1]:]
    response = eval_tokenizer.decode(response_ids[0], skip_special_tokens=True)
    response = truncate_generated_code(response)

    stats = run_python_with_test_stats(
        code=response,
        setup_code=example.test_setup_code,
        tests=example.test_list,
    )

    aggregate_results.append(
        {
            "task_id": example.task_id,
            "passed_count": stats["passed_count"],
            "total_count": stats["total_count"],
            "strict_pass": stats["passed_count"] == stats["total_count"],
        }
    )

num_tasks = len(aggregate_results)
strict_pass_rate = sum(r["strict_pass"] for r in aggregate_results) / num_tasks
any_pass_rate = sum(r["passed_count"] > 0 for r in aggregate_results) / num_tasks
mean_test_pass_fraction = sum(r["passed_count"] / r["total_count"] for r in aggregate_results) / num_tasks

summary = {
    "num_tasks": num_tasks,
    "strict_pass_rate": strict_pass_rate,
    "any_pass_rate": any_pass_rate,
    "mean_test_pass_fraction": mean_test_pass_fraction,
}
summary


{'num_tasks': 32,
 'strict_pass_rate': 0.5,
 'any_pass_rate': 0.75,
 'mean_test_pass_fraction': 0.6354166666666666}

In [17]:
import json
from pathlib import Path

def run_variant(
    *,
    name: str,
    use_continuous_batching: bool,
    use_speculative_decoding: bool,
    init_adapter_path: str,
    output_dir: str,
    steps: int = 3,
    batch_size: int = 1,
    group_size: int = 2,
    max_new_tokens: int = 192,
    draft_model_name: str | None = None,
):
    config = GRPOConfig(
        model_name=MODEL_NAME,
        steps=steps,
        batch_size=batch_size,
        group_size=group_size,
        max_new_tokens=max_new_tokens,
        temperature=0.8,
        top_p=0.95,
        output_dir=output_dir,
        init_adapter_path=init_adapter_path,
        use_continuous_batching=use_continuous_batching,
        use_speculative_decoding=use_speculative_decoding,
        draft_model_name=draft_model_name,
    )

    trainer = GRPOTrainer(
        config=config,
        environment=task.environment,
        verifier=task.verifier,
    )
    history = trainer.train()

    metrics_path = Path(output_dir) / "metrics.jsonl"
    rows = [
        json.loads(line)
        for line in metrics_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    def mean_of(key: str) -> float:
        vals = [float(row.get(key, 0.0)) for row in rows]
        return sum(vals) / max(1, len(vals))

    summary = {
        "name": name,
        "steps": len(rows),
        "mean_reward": mean_of("mean_reward"),
        "mean_total_step_time_ms": mean_of("total_step_time_ms"),
        "mean_generation_time_ms": mean_of("generation_time_ms"),
        "mean_training_time_ms": mean_of("training_time_ms"),
        "mean_tokens_per_second": mean_of("tokens_per_second"),
        "mean_rollout_peak_vram_mb": mean_of("rollout_peak_vram_mb"),
        "mean_runtime_headroom_mb": mean_of("rollout_runtime_headroom_mb"),
        "mean_scheduler_decode_passes": mean_of("scheduler_decode_passes"),
        "mean_scheduler_prefill_passes": mean_of("scheduler_prefill_passes"),
        "mean_paged_kv_pressure": mean_of("paged_kv_allocator_pressure"),
        "mean_kv_pressure": mean_of("scheduler_decode_kv_pressure"),
    }
    return summary, history, rows


In [18]:
continuous_summary, continuous_history, continuous_rows = run_variant(
    name="continuous",
    use_continuous_batching=True,
    use_speculative_decoding=False,
    init_adapter_path=BOOTSTRAP_DIR,
    output_dir="./mbpp_rl_continuous",
)
continuous_summary


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

step=0 | mean_reward=0.0500 | reward_std=0.0000 | policy_loss=-0.0000 | kl_loss=0.0008 | mean_token_kl=0.0773 | beta=0.0100 | total_loss=0.0008 | learning_rate=0.0000 | generation_time_ms=16443.5129 | prefill_time_ms=150.7835 | decode_time_ms=15187.3783 | training_time_ms=302.5256 | generation_peak_vram_mb=7442.1606 | rollout_peak_vram_mb=7442.1606 | generation_runtime_headroom_mb=15120.9644 | rollout_runtime_headroom_mb=15120.9644 | peak_vram_mb=8571.4155 | tokens_per_second=22.9308 | prefill_tokens_per_second=1923.2876 | decode_tokens_per_second=25.2842 | padding_ratio=0.0001 | padding_waste_tokens=12.0000 | cache_reuse_effectiveness=0.9969 | unique_response_ratio=1.0000 | advantage_mean=0.0 | advantage_std=0.0 | training_peak_vram_mb=8571.41552734375 | training_runtime_headroom_mb=13991.70947265625 | total_step_time_ms=16746.038477000184 | phase_a_fraction=0.9819344990508869 | effective_batch_tokens=384.0 | generation_padding_ratio=0.0 | generation_padding_waste_tokens=0.0 | sequenc

step=1 | mean_reward=1.0000 | reward_std=0.0000 | policy_loss=-0.0000 | kl_loss=0.0007 | mean_token_kl=0.0680 | beta=0.0100 | total_loss=0.0007 | learning_rate=0.0000 | generation_time_ms=15935.3073 | prefill_time_ms=148.2376 | decode_time_ms=14674.2210 | training_time_ms=291.4518 | generation_peak_vram_mb=8571.4155 | rollout_peak_vram_mb=8571.4155 | generation_runtime_headroom_mb=13991.7095 | rollout_runtime_headroom_mb=13991.7095 | peak_vram_mb=8638.1792 | tokens_per_second=23.6030 | prefill_tokens_per_second=1956.3192 | decode_tokens_per_second=26.1683 | padding_ratio=0.0001 | padding_waste_tokens=13.0000 | cache_reuse_effectiveness=0.9969 | unique_response_ratio=1.0000 | advantage_mean=0.0 | advantage_std=0.0 | training_peak_vram_mb=8638.17919921875 | training_runtime_headroom_mb=13924.94580078125 | total_step_time_ms=16226.759089000097 | phase_a_fraction=0.9820388141956418 | effective_batch_tokens=383.0 | generation_padding_ratio=0.0 | generation_padding_waste_tokens=0.0 | sequenc

{'name': 'continuous',
 'steps': 3,
 'mean_reward': 0.3750000012417634,
 'mean_total_step_time_ms': 16317.571081666756,
 'mean_generation_time_ms': 16015.92335466671,
 'mean_training_time_ms': 301.64772700004505,
 'mean_tokens_per_second': 23.479597235584095,
 'mean_rollout_peak_vram_mb': 8217.251790364584,
 'mean_runtime_headroom_mb': 14345.873209635416,
 'mean_scheduler_decode_passes': 192.0,
 'mean_scheduler_prefill_passes': 1.0,
 'mean_paged_kv_pressure': 0.0,
 'mean_kv_pressure': 0.4697265625}

In [20]:
standard_summary, standard_history, standard_rows = run_variant(
    name="standard",
    use_continuous_batching=False,
    use_speculative_decoding=False,
    init_adapter_path=BOOTSTRAP_DIR,
    output_dir="./mbpp_rl_standard",
)
standard_summary


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

step=0 | mean_reward=0.0500 | reward_std=0.0000 | policy_loss=-0.0000 | kl_loss=0.0008 | mean_token_kl=0.0822 | beta=0.0100 | total_loss=0.0008 | learning_rate=0.0000 | generation_time_ms=25104.0183 | prefill_time_ms=0.0000 | decode_time_ms=24835.5372 | training_time_ms=298.3967 | generation_peak_vram_mb=9132.1953 | rollout_peak_vram_mb=9132.1953 | generation_runtime_headroom_mb=13430.9297 | rollout_runtime_headroom_mb=13430.9297 | peak_vram_mb=11989.9258 | tokens_per_second=15.0379 | prefill_tokens_per_second=0.0000 | decode_tokens_per_second=15.4617 | padding_ratio=0.0203 | padding_waste_tokens=14.0000 | cache_reuse_effectiveness=0.0000 | unique_response_ratio=1.0000 | advantage_mean=0.0 | advantage_std=0.0 | training_peak_vram_mb=11989.92578125 | training_runtime_headroom_mb=10573.19921875 | total_step_time_ms=25402.41505699987 | phase_a_fraction=0.9882532147699226 | effective_batch_tokens=382.0 | generation_padding_ratio=0.0 | generation_padding_waste_tokens=0.0 | sequence_padding_

{'name': 'standard',
 'steps': 3,
 'mean_reward': 0.21666665996114412,
 'mean_total_step_time_ms': 25199.711566999973,
 'mean_generation_time_ms': 24912.245039666686,
 'mean_training_time_ms': 287.4665273332842,
 'mean_tokens_per_second': 15.160132719190706,
 'mean_rollout_peak_vram_mb': 11059.6865234375,
 'mean_runtime_headroom_mb': 11503.4384765625,
 'mean_scheduler_decode_passes': 0.0,
 'mean_scheduler_prefill_passes': 0.0,
 'mean_paged_kv_pressure': 0.0,
 'mean_kv_pressure': 0.0}

In [26]:
 print("Standard mean reward:", standard_summary["mean_reward"])
print("Continuous mean reward:", continuous_summary["mean_reward"])
print("Reward delta:", continuous_summary["mean_reward"] - standard_summary["mean_reward"])


Standard mean reward: 0.21666665996114412
Continuous mean reward: 0.3750000012417634
Reward delta: 0.1583333412806193


In [27]:
import pandas as pd

systems_df = pd.DataFrame([
    {
        "mode": "standard",
        "mean_step_time_ms": standard_summary["mean_total_step_time_ms"],
        "mean_generation_time_ms": standard_summary["mean_generation_time_ms"],
        "mean_training_time_ms": standard_summary["mean_training_time_ms"],
        "tokens_per_second": standard_summary["mean_tokens_per_second"],
        "rollout_peak_vram_mb": standard_summary["mean_rollout_peak_vram_mb"],
        "runtime_headroom_mb": standard_summary["mean_runtime_headroom_mb"],
        "scheduler_decode_passes": standard_summary["mean_scheduler_decode_passes"],
        "scheduler_prefill_passes": standard_summary["mean_scheduler_prefill_passes"],
        "kv_pressure": standard_summary["mean_kv_pressure"],
        "mean_reward": standard_summary["mean_reward"],
    },
    {
        "mode": "continuous",
        "mean_step_time_ms": continuous_summary["mean_total_step_time_ms"],
        "mean_generation_time_ms": continuous_summary["mean_generation_time_ms"],
        "mean_training_time_ms": continuous_summary["mean_training_time_ms"],
        "tokens_per_second": continuous_summary["mean_tokens_per_second"],
        "rollout_peak_vram_mb": continuous_summary["mean_rollout_peak_vram_mb"],
        "runtime_headroom_mb": continuous_summary["mean_runtime_headroom_mb"],
        "scheduler_decode_passes": continuous_summary["mean_scheduler_decode_passes"],
        "scheduler_prefill_passes": continuous_summary["mean_scheduler_prefill_passes"],
        "kv_pressure": continuous_summary["mean_kv_pressure"],
        "mean_reward": continuous_summary["mean_reward"],
    },
])

systems_df


,mode,mean_step_time_ms,mean_generation_time_ms,mean_training_time_ms,tokens_per_second,rollout_peak_vram_mb,runtime_headroom_mb,scheduler_decode_passes,scheduler_prefill_passes,kv_pressure,mean_reward
0,standard,25199.711567,24912.245040,287.466527,15.160133,11059.686523,11503.438477,0.0,0.0,0.000000,0.216667
1,continuous,16317.571082,16015.923355,301.647727,23.479597,8217.251790,14345.873210,192.0,1.0,0.469727,0.375000


In [28]:
systems_df_rounded = systems_df.copy()
for col in systems_df_rounded.columns:
    if col != "mode":
        systems_df_rounded[col] = systems_df_rounded[col].map(lambda x: round(float(x), 3))
systems_df_rounded


,mode,mean_step_time_ms,mean_generation_time_ms,mean_training_time_ms,tokens_per_second,rollout_peak_vram_mb,runtime_headroom_mb,scheduler_decode_passes,scheduler_prefill_passes,kv_pressure,mean_reward
0,standard,25199.712,24912.245,287.467,15.16,11059.687,11503.438,0.0,0.0,0.00,0.217
1,continuous,16317.571,16015.923,301.648,23.48,8217.252,14345.873,192.0,1.0,0.47,0.375


In [29]:
systems_verdict = {
    "step_time_speedup_x": standard_summary["mean_total_step_time_ms"] / continuous_summary["mean_total_step_time_ms"],
    "generation_speedup_x": standard_summary["mean_generation_time_ms"] / continuous_summary["mean_generation_time_ms"],
    "throughput_gain_x": continuous_summary["mean_tokens_per_second"] / standard_summary["mean_tokens_per_second"],
    "rollout_vram_reduction_mb": standard_summary["mean_rollout_peak_vram_mb"] - continuous_summary["mean_rollout_peak_vram_mb"],
    "headroom_gain_mb": continuous_summary["mean_runtime_headroom_mb"] - standard_summary["mean_runtime_headroom_mb"],
    "reward_delta": continuous_summary["mean_reward"] - standard_summary["mean_reward"],
}
systems_verdict


{'step_time_speedup_x': 1.5443298172797635,
 'generation_speedup_x': 1.5554672988871274,
 'throughput_gain_x': 1.5487725385056859,
 'rollout_vram_reduction_mb': 2842.434733072916,
 'headroom_gain_mb': 2842.434733072916,
 'reward_delta': 0.1583333412806193}